# FAZ A2 -- EC bolge taramasi, uzatilmis `n` + ogrenme seviyesi olcutuFaz A (`region_scan_colab.ipynb`) sonucu: **bolge listesi bos**, ama Katman BA0 fidelity kernel icin ustel konsantrasyonu kesin gosterdi (`Var ~ 4^-n`,`mu = 2^-n`, egim -1.422 vs Haar limiti -1.386, `R^2 = 1.000`).Liste bos cikmasinin sebebi ESIK KALIBRASYONU: binom tahmin edici icin kapali form`S_req = 2^n` -> `n=12`'de 4096, kilit esigi `10^5` ise `n >~ 17` gerektiriyor.**Bu defter esigi DUSURMEZ** (sonuca bakip esik oynatmak, "esigi verinize goresectiniz" itirazina acik olurdu). Bunun yerine iki degisiklik:### (a) `n` kesismenin otesine uzatildi -- `H` seri ICINDE sabit`H`'yi `n` ile kucultmek KARISTIRICI olurdu: `n` boyunca egim uydururken havuz dadegisirse "n buyudu" ile "havuz kesildi" etkileri ayirt edilemez. Bunun yerine **ucayri seri**, her biri kendi icinde sabit `H`:| Seri | `n` araligi ||---|---|| `H=2` | 4 … 18 || `H=3` | 4 … 16 || `H=4` | 4 … 14 || A0 fidelity (referans) | 4 … 20 |Bonus: uc seriyi yan yana koymak, egimin `H` ile nasil dikleştigini -- yani globalityekseninin nicel katkisini -- `n`-taramasinin **icinden** verir.### (b) Katman A ikiye ayrildiFaz A'daki olcut sadece **olcum** maliyetini yakaliyordu. Ama `mu = 2^-n` demek`K -> I` demek: sonsuz shot'ta bile kernel ridge genelleyemez. Bu **ogrenmeseviyesinde EC** ve `n<=12`'de zaten aktif olabilir.| Olcut | Tanim | Esik ||---|---|---|| **A-shot** | `S_req = sigma^2_shot / Var[k~]` | `> 10^5` (Faz A ile ayni, DEGISMEDI) || **A-learn** | `effrank(K)/M` | `> 0.90` |`A-learn` gerekcesi: `K -> I` tam olarak `effrank/M -> 1` demektir; `effrank/M > 0.9`,kernel'in birim matristen %10'dan az saptigi, yani ogrenilecek dusuk-boyutlu yapikalmadigi anlamina gelir. **Esik sonuc gorulmeden burada sabitlenmistir.**> **Acik durustluk notu.** "`w<=2` kernel konsantre olmuyor" tek basina UCUZ bir> sonuctur -- sabit bir kernel de konsantre olmaz. Bu bulgu ancak Faz B'de ayni> kernel'in HALA OGRENDIGI gosterilirse anlam kazanir (Katman D). Bu defter tek> basina bir iddia kurmaz.

## Bolum 0 - Kurulum

In [ ]:
import os, json, time, math
from math import comb
from itertools import product
import numpy as np
import matplotlib.pyplot as plt

try:
    import cupy as cp
    xp, GPU = cp, True
    print('GPU:', cp.cuda.runtime.getDeviceProperties(0)['name'].decode(),
          f"{cp.cuda.Device(0).mem_info[1]/2**30:.1f} GB")
except Exception as e:
    xp, GPU = np, False
    print('GPU YOK, CPU ile devam (cok yavas olacak):', e)

CDT = xp.complex64 if GPU else xp.complex128
FDT = xp.float32   if GPU else xp.float64
to_xp = lambda a, dtype=None: xp.asarray(a, dtype=dtype)
to_np = lambda a: cp.asnumpy(a) if GPU and isinstance(a, cp.ndarray) else np.asarray(a)
def free_mem():
    if GPU: cp.get_default_memory_pool().free_all_blocks()

## Bolum 1 - On-kayitli sabitler**Faz A ile ayni olanlar DEGISTIRILMEDI.** Yeni olan sadece `EFFRANK_GATE` ve `n` araliklari.

In [ ]:
SHOT_BUDGETS = [1e3, 1e5, 1e7]
LOCK_BUDGET  = 1e5      # Faz A ile AYNI -- sonuca bakip oynatilmadi
BW           = 0.3      # Faz A ile AYNI
MEAN_GATE    = 0.5      # Katman C: offdiag_mean >= bu ise K->11^T (bandwidth cokusu)
EFFRANK_GATE = 0.90     # YENI: Katman A-learn esigi (sonuc gorulmeden sabitlendi)
SEED         = 0

# Seri basina n araligi. H seri ICINDE SABIT -- karistirici yok.
SERIES   = {2: [4,6,8,10,12,14,16,18], 3: [4,6,8,10,12,14,16], 4: [4,6,8,10,12,14]}
N_FID    = [4,6,8,10,12,14,16,18,20]      # A0 referansi (Pauli havuzu gerekmez, ucuz)
DEPTHS   = [1, 2, 4, 8]
ENTS     = [0.0, 0.25, 0.5, 1.0]
PS       = [0.0, 0.05]
REF      = dict(depth=8, ent=1.0, p=0.0)  # en guclu EC kosesi

def M_of(n):                              # bellek icin nokta sayisi n ile kisilir
    return 400 if n <= 12 else (250 if n <= 16 else 200)

CKPT = 'phase_a2_ckpt'; os.makedirs(CKPT, exist_ok=True)

print(f'{"n":>3} {"dim":>9} {"M":>4} | ' + ' '.join(f'H={h}:{"":>7}' for h in SERIES))
for n in N_FID:
    row = f'{n:>3} {2**n:>9} {M_of(n):>4} | '
    for h in SERIES:
        row += (f'H={h}:{sum(comb(n,k)*3**k for k in range(h+1)):>7} '
                if n in SERIES[h] else f'H={h}:{"-":>7} ')
    print(row)
print(f'\nToplam hucre: {sum(len(v) for v in SERIES.values())*len(DEPTHS)*len(ENTS)*len(PS)}'
      f' + {len(N_FID)*len(DEPTHS)*len(ENTS)} A0 referansi')

## Bolum 2/3 - Cekirdek + teshis  (kaynak: `src/qk_core.py`, TEK KAYNAK)

In [ ]:
"""Kuantum kernel cekirdegi -- TEK KAYNAK.

Bu dosya defterlere OLDUGU GIBI GOMULUR (build_*_nb.py `open(...).read()` ile alir),
boylece Colab'de self-contained kalir ama kod tek yerde durur.

BEKLENEN ORTAM: `xp`, `GPU`, `CDT`, `FDT`, `to_xp`, `to_np` defterin 0. hucresinde
tanimlanmis olmali (cupy/numpy dagitimi).

Dogrulanmis (bkz. defterlerin Bolum 4):
  fwht == H^{otimes n} · pauli_expect_fast == naif yol (2.2e-16) · Parseval ·
  GFQK_{w<=n} == fidelity kernel · dort eksende S_req monoton · Katman B/C ayrimlari
"""


# ============================================================================
# DEVRE -- yogun 2^n x 2^n matris KURULMAZ
# ============================================================================

def fwht(a, normalize=True):
    """Son eksende H^{otimes n}. a: (..., 2^n). O(N n 2^n), yogun matris yok."""
    dim = a.shape[-1]
    n = int(np.log2(dim))
    lead = a.shape[:-1]
    b = a.reshape(*lead, *([2] * n))
    for ax in range(len(lead), len(lead) + n):
        x = xp.moveaxis(b, ax, -1)
        b = xp.moveaxis(xp.stack([x[..., 0] + x[..., 1], x[..., 0] - x[..., 1]], -1), -1, ax)
    b = b.reshape(*lead, dim)
    return b * (2.0 ** (-n / 2)) if normalize else b


def sign_tables(n):
    z = np.arange(2 ** n)
    S = 1.0 - 2.0 * ((z[:, None] >> np.arange(n - 1, -1, -1)[None, :]) & 1)
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    SP = np.stack([S[:, i] * S[:, j] for i, j in pairs], 1) if pairs else np.zeros((2 ** n, 0))
    return S, SP, pairs


def encode(X, n, depth, ent, bw, norm='none'):
    """ZZ-feature map, `depth` tekrar. Doner: (M, 2^n) statevector.

    faz(z) = -( bw * sum_i x_i s_i(z)  +  ent * bw^2 * sum_{i<j} x_i x_j s_ij(z) )

    `norm` -- `n`-BAGIMLI KONFOUND kontrolu:
      'none' : ham. Terim sayilari n ve n(n-1)/2 oldugu icin faz sacilimi
               sirasiyla ~sqrt(n) ve ~n buyur -> BUYUK n'de devre DAHA COK karisir.
               Bu, "n buyudu" ile "encoding guclendi" etkilerini karistirir.
      'sqrt' : her iki terim kendi terim sayisinin karekokune bolunur -> faz
               sacilimi n'den BAGIMSIZ. "Ayni harita, farkli n" yorumu gecerli olur.

    ent=0 -> faz tamamen tek-qubit -> durum CARPIM kalir (Meyer-Wallach tam 0),
    `norm` bunu degistirmez (tek-qubit terim carpimsal yapiyi bozmaz).
    """
    S, SP, pairs = sign_tables(n)
    Xp = np.stack([X[:, i] * X[:, j] for i, j in pairs], 1) if pairs else np.zeros((len(X), 0))
    c1, c2 = 1.0, 1.0
    if norm == 'sqrt':
        c1 = 1.0 / np.sqrt(n)
        c2 = 1.0 / np.sqrt(max(len(pairs), 1))
    elif norm != 'none':
        raise ValueError(norm)
    phase = -(c1 * bw * (to_xp(X, FDT) @ to_xp(S, FDT).T)
              + c2 * ent * (bw ** 2) * (to_xp(Xp, FDT) @ to_xp(SP, FDT).T))
    D = xp.exp(1j * phase).astype(CDT)
    psi = xp.zeros((len(X), 2 ** n), dtype=CDT)
    psi[:, 0] = 1.0
    for _ in range(depth):
        psi = fwht(psi) * D
    return psi


def meyer_wallach(psi, n):
    """Q = 2(1 - (1/n) sum_k tr(rho_k^2)).  0 = carpim, 1 = maksimal dolanik."""
    M = psi.shape[0]
    tot = xp.zeros(M, dtype=FDT)
    for k in range(n):
        a = xp.moveaxis(psi.reshape(M, *([2] * n)), k + 1, 1).reshape(M, 2, -1)
        rho = xp.einsum('mir,mjr->mij', a, a.conj())
        tot += xp.real(xp.einsum('mij,mji->m', rho, rho)).astype(FDT)
    return float(to_np(xp.mean(2.0 * (1.0 - tot / n))))


# ============================================================================
# PAULI DOMENI -- flip-maskesi + Walsh-Hadamard hizli yolu
# ============================================================================

def _popcount(v):
    v = np.asarray(v, dtype=np.int64).copy()
    r = np.zeros_like(v)
    while np.any(v):
        r += v & 1
        v >>= 1
    return r


def _spread4(v, n):
    v = np.asarray(v, dtype=np.int64)
    out = np.zeros_like(v)
    for k in range(n):
        out += ((v >> k) & 1) * (4 ** k)
    return out


def pool_layout(n, h):
    """Doner: (flip_maskeleri, [(g_dizisi, pauli_indeksleri) her f icin],
               tum_indeksler_sirali, tum_agirliklar_sirali).  Havuz: agirlik <= h."""
    allz = np.arange(2 ** n)
    pc = _popcount(allz)
    fs = np.where(pc <= h)[0]
    blocks, idx_all, w_all = [], [], []
    for f in fs:
        g = np.where(pc[allz | f] <= h)[0]            # agirlik = |f OR g|
        pidx = 2 * _spread4(f & g, n) + _spread4(f & ~g, n) + 3 * _spread4(g & ~f, n)
        blocks.append((g, pidx))
        idx_all.append(pidx)
        w_all.append(pc[g | f])
    idx_all = np.concatenate(idx_all)
    w_all = np.concatenate(w_all)
    o = np.argsort(idx_all)
    return fs, blocks, idx_all[o], w_all[o]


def pauli_expect_fast(psi, n, layout):
    """<P_{f,g}> = Re[ (-i)^{|f&g|} * WHT_z( conj(psi_z) psi_{z^f} )[g] ]

    Ayni f'yi paylasan tum operatorler tek gather + tek WHT ile hesaplanir.
    n=12, w<=4: 46 666 operator yerine 794 donusum.
    """
    fs, blocks, _, _ = layout
    z = xp.arange(2 ** n)
    conj = psi.conj()
    cols, keys = [], []
    for f, (g, pidx) in zip(fs, blocks):
        Wt = fwht(conj * psi[:, z ^ int(f)], normalize=False)
        ph = ((-1j) ** to_xp(_popcount(np.bitwise_and(int(f), g)))).astype(CDT)
        cols.append(xp.real(Wt[:, to_xp(g.astype(np.int64))] * ph[None, :]).astype(FDT))
        keys.append(pidx)
    E = xp.concatenate(cols, axis=1)
    return E[:, to_xp(np.argsort(np.concatenate(keys)))]


def pauli_expect_naive(psi, n, idx):
    """Referans yol -- SADECE dogrulama icin (yavas)."""
    flip = np.zeros(len(idx), np.int64)
    phm = np.zeros(len(idx), np.int64)
    nY = np.zeros(len(idx), np.int64)
    for k in range(n):
        d = (np.asarray(idx) // (4 ** k)) % 4
        flip |= np.where((d == 1) | (d == 2), 1 << k, 0)
        phm |= np.where((d == 2) | (d == 3), 1 << k, 0)
        nY += (d == 2)
    z = np.arange(2 ** n, dtype=np.int64)
    P = to_np(psi)
    out = np.zeros((P.shape[0], len(idx)))
    for t in range(len(idx)):
        zf = z ^ flip[t]
        sgn = 1.0 - 2.0 * (_popcount(zf & phm[t]) & 1)
        out[:, t] = np.real(np.sum(P.conj() * sgn[None, :] * P[:, zf], 1) * (1j) ** int(nY[t]))
    return out


# ============================================================================
# KERNEL -- birim kosegene normalize + tahmin edicinin shot varyansi
# ============================================================================

def kernel_gfqk(E_pool, w_pool, H, p):
    """GFQK_{w<=H}, gurultulu, birim kosegene normalize. Doner (K~, sigma2_shot).

    Gurultu: qubit basina depolarizing, Pauli-transfer ozdegeri lam=1-4p/3 -> EGZAKT.
    Shot varyansi: classical shadow, operator basina 3^w.
      k = sum_i a_i b_i, Var(a^_i) ~ 3^{w_i}
      -> Var(k^) ~ 2 sum_i 3^{w_i} <psi~_i^2>,   <psi~_i^2> = mean(E[:,i]^2)
    `c`'nin BIRINCI kuvveti kullanilir; c^2 yazmak pay/paydada sadelesip gurultu
    eksenini ters cevirir (bkz. 02_PROJECT_STATUS.md hata #16).
    """
    w = np.asarray(w_pool)
    m = to_xp(w <= H)
    ww = to_xp(w[w <= H].astype(np.float64), FDT)
    E = E_pool[:, m] * xp.sqrt((1.0 - 4.0 * p / 3.0) ** (2 * ww))[None, :]
    K = E @ E.T
    dg = xp.maximum(xp.diag(K), 1e-30)
    s2 = float(to_np(2.0 * xp.sum((3.0 ** ww) * xp.mean(E ** 2, 0)) / xp.mean(dg) ** 2))
    return K / xp.sqrt(xp.outer(dg, dg)), s2


def kernel_fidelity(psi):
    """A0 referansi. diag zaten 1; tahmin edici compute-uncompute BINOM -> sigma2 None.
    Bu, GFQK'dan FARKLI bir olcum protokoludur; mutlak shot sayilari ayni para birimi
    degil (bkz. hata #17)."""
    return xp.abs(psi.conj() @ psi.T) ** 2, None


# ============================================================================
# TESHIS -- Katman A-shot / A-learn / B / C
# ============================================================================

def effrank(ev):
    ev = xp.clip(ev.real, 0, None)
    s1, s2 = ev.sum(), (ev ** 2).sum()
    return float(to_np(s1 * s1 / s2)) if float(to_np(s2)) > 0 else 0.0


def diagnose(K, sigma2_shot=None, budgets=(1e3, 1e5, 1e7)):
    """S_req = sigma2_shot / Var[k~]  --  `1/Var` DEGIL (hata #15)."""
    M = K.shape[0]
    off = K[~xp.eye(M, dtype=bool)]
    var = float(to_np(off.var()))
    mu = float(to_np(off.mean()))
    kc = xp.clip(off, 0.0, 1.0)
    s2 = float(to_np(xp.mean(kc * (1 - kc)))) if sigma2_shot is None else float(sigma2_shot)
    er = effrank(xp.linalg.eigvalsh(K))
    d = dict(offdiag_var=var, offdiag_mean=mu, sigma2_shot=s2, M=M,
             s_req=(s2 / var) if var > 0 else float('inf'),
             effrank_K=er, effrank_frac=er / M)
    for S in budgets:
        d[f'resolv@{S:.0e}'] = float(to_np(xp.mean(xp.abs(off - mu) > 3 * np.sqrt(s2 / S))))
    return d


def layers_ac(d, budget, mean_gate, effrank_gate):
    """Doner (durum, a_shot, a_learn). Katman C ONCE: K->11^T ise bandwidth cokusu, EC DEGIL.

    UYARI: a_learn (effrank/M -> 1) EC'ye OZGU DEGILDIR -- carpim durumlarinda da
    tetiklenir ve n>=14'te 1.0000'da doyar. Bolge listesi Katman B (ustel onayi) ile
    KAPILANMALIDIR (bkz. 02_PROJECT_STATUS.md 6.96).
    """
    if d['offdiag_mean'] >= mean_gate:
        return 'COLLAPSE', False, False
    a_shot = d['s_req'] > budget
    a_learn = d['effrank_frac'] > effrank_gate
    if a_shot and a_learn:
        return 'EC-BOTH', True, True
    if a_shot:
        return 'EC-SHOT', True, False
    if a_learn:
        return 'EC-LEARN', False, True
    return 'NO-EC', False, False


def layer_b(ns, vals, n_boot=2000, seed=0):
    """log V ~ a + b*n (ustel)  vs  log V ~ a' + b'*log n (polinom), AIC ile.
    `exponential` = egim GA ust siniri < 0  VE  AIC ustelde daha kucuk.

    UYARI: `exponential=False`, "EC yok" DEMEK DEGILDIR -- egri henuz bukulme
    bolgesinde olabilir (asimptota girmemis). Bkz. 6.96 acik #1.
    """
    ns = np.asarray(ns, float)
    v = np.log(np.maximum(np.asarray(vals, float), 1e-300))
    k = len(ns)
    if k < 3:
        return None

    def fit(x, y):
        b, a = np.polyfit(x, y, 1)
        r = y - (a + b * x)
        ss = ((y - y.mean()) ** 2).sum()
        return b, (1 - (r ** 2).sum() / ss if ss > 0 else 0.0), (r ** 2).sum()

    b_e, r2_e, ss_e = fit(ns, v)
    b_p, r2_p, ss_p = fit(np.log(ns), v)
    aic = lambda ss: k * np.log(max(ss, 1e-300) / k) + 4
    rng = np.random.default_rng(seed)
    bs = []
    while len(bs) < n_boot:                       # dejenere cekilis (tek n) egimi bozar
        i = rng.integers(0, k, k)
        if len(np.unique(ns[i])) >= 2:
            bs.append(np.polyfit(ns[i], v[i], 1)[0])
    bs = np.array(bs)
    return dict(slope=float(b_e), slope_lo=float(np.percentile(bs, 2.5)),
                slope_hi=float(np.percentile(bs, 97.5)), r2_exp=float(r2_e),
                r2_poly=float(r2_p), d_aic=float(aic(ss_e) - aic(ss_p)),
                exponential=bool(np.percentile(bs, 97.5) < 0 and aic(ss_e) < aic(ss_p)))


# --- defter-ozel ince sarmalayicilar (sabitleri baglar) ---
def enc(X, n, depth, ent, norm='none'):
    return encode(X, n, depth, ent, bw=BW, norm=norm)

def diag_(K, s2=None):
    return diagnose(K, s2, budgets=tuple(SHOT_BUDGETS))

def layers_(d):
    return layers_ac(d, LOCK_BUDGET, MEAN_GATE, EFFRANK_GATE)

## Bolum 4 - DOGRULAMA (gecmezse ASAGI DEVAM ETME)

In [ ]:
def _validate():
    rng = np.random.default_rng(0); n = 4
    X = rng.uniform(0, 2*np.pi, size=(24, n))
    Hm = np.array([[1,1],[1,-1]], float)/np.sqrt(2); Hn = Hm
    for _ in range(n-1): Hn = np.kron(Hn, Hm)
    v = rng.normal(size=(5, 2**n))+1j*rng.normal(size=(5, 2**n))
    assert np.abs(to_np(fwht(to_xp(v, CDT))) - v @ Hn.T).max() < 1e-5
    print('[1a] OK  FWHT == H^{otimes n}')

    psi = enc(X, n, 2, 1.0); lay = pool_layout(n, n)
    err = np.abs(to_np(pauli_expect_fast(psi, n, lay)) - pauli_expect_naive(psi, n, lay[2])).max()
    assert err < 1e-4, f'hizli yol saptI: {err:.2e}'
    print(f'[1b] OK  hizli Pauli yolu == naif yol (maxerr {err:.2e})')

    for d in DEPTHS:
        assert meyer_wallach(enc(X, n, d, 0.0), n) < 1e-5, f'ent=0 depth={d} dolanik!'
    mw = [meyer_wallach(enc(X, n, 2, e), n) for e in ENTS]
    assert all(a < b for a, b in zip(mw, mw[1:])), mw
    print(f'[2] OK  ent ekseni temiz ve monoton  MW={[round(x,3) for x in mw]}')

    E = pauli_expect_fast(psi, n, lay)
    assert np.abs(to_np(xp.sum(E**2, 1))/2**n - 1).max() < 1e-4, 'Parseval bozuk'
    Kp, _ = kernel_gfqk(E, lay[3], n, 0.0); Kf, _ = kernel_fidelity(psi)
    assert np.abs(to_np(Kp-Kf)).max() < 1e-4
    print('[3] OK  Parseval + GFQK_{w<=n} == tam fidelity kernel')

    # A-learn olcutu: K=I -> frac=1, K=11^T -> frac=1/M
    M = 50
    assert abs(diag_(xp.eye(M, dtype=FDT))['effrank_frac'] - 1.0) < 1e-6
    assert abs(diag_(xp.ones((M, M), dtype=FDT))['effrank_frac'] - 1.0/M) < 1e-6
    print('[4] OK  effrank_frac: K=I -> 1.000, K=11^T -> 1/M')

    assert layers_(dict(offdiag_mean=.98, s_req=1e9, effrank_frac=.99))[0] == 'COLLAPSE'
    assert layers_(dict(offdiag_mean=.01, s_req=1e9, effrank_frac=.99))[0] == 'EC-BOTH'
    assert layers_(dict(offdiag_mean=.01, s_req=1e2, effrank_frac=.99))[0] == 'EC-LEARN'
    assert layers_(dict(offdiag_mean=.01, s_req=1e9, effrank_frac=.10))[0] == 'EC-SHOT'
    assert layers_(dict(offdiag_mean=.01, s_req=1e2, effrank_frac=.10))[0] == 'NO-EC'
    print('[5] OK  bes durum da dogru siniflaniyor (C once, sonra A-shot/A-learn)')

    nv = np.array([4,6,8,10,12], float)
    assert layer_b(nv, np.exp(-1.4*nv))['exponential'] is True
    assert layer_b(nv, nv**-3.0)['exponential'] is False
    print('[6] OK  Katman B ustel/polinom ayrimi')

    # H seri icinde sabit mi (karistirici kontrolu)
    for h, ns_ in SERIES.items():
        assert len(ns_) >= 5, f'H={h} serisinde egim icin yeterli n yok'
    print('[7] OK  her seri >=5 n-noktasi, H seri icinde sabit')

    # [8] effrank_frac M'e bagli mi?  Hakem itirazi: "olcut M ile saturate oluyor".
    #     Karsilastirma AYNI M'de yapildigi icin adil, ama M-duyarliligini yine de
    #     olcup raporluyoruz. A0 ~1'de, GFQK dusukte kalmali (ikisi de her M'de).
    n8 = 8; lay8 = pool_layout(n8, 3)
    print('[8] effrank/M -- M duyarliligi (n=8, depth=8, ent=1.0):')
    for Mt in [100, 200, 400]:
        Xt = np.random.default_rng(11).uniform(0, 2*np.pi, size=(Mt, n8))
        pt = enc(Xt, n8, 8, 1.0)
        f0 = diag_(kernel_fidelity(pt)[0])['effrank_frac']
        Kg, s2g = kernel_gfqk(pauli_expect_fast(pt, n8, lay8), lay8[3], 3, 0.0)
        fg = diag_(Kg, s2g)['effrank_frac']
        print(f'     M={Mt:>4}  A0={f0:.3f}   GFQK w<=3={fg:.3f}   fark={f0-fg:+.3f}')
        assert f0 - fg > 0.5, 'A0 ile GFQK ayrimi bu M\'de kayboldu'
    print('     OK  ayrim her M\'de korunuyor -> olcut M artefakti degil')
    print('\nTUM DOGRULAMALAR GECTI')

_validate()

## Bolum 5 - KosuIki gecis:1. **A0 fidelity referansi** -- `n` 4…20, Pauli havuzu gerekmez, ucuz2. **GFQK serileri** -- her `H` icin kendi `n` araligi, `H` seri icinde SABITHer `(seri, n)` bitince checkpoint yazilir; kesilirse kaldigi yerden devam eder.

In [ ]:
def ck(tag):
    return os.path.join(CKPT, f'{tag}.json')

def run_fidelity():
    out = {}
    for n in N_FID:
        p = ck(f'fid_n{n}')
        if os.path.exists(p):
            out[n] = json.load(open(p)); print(f'  fid n={n}: checkpoint'); continue
        t0 = time.time(); M = M_of(n)
        X = np.random.default_rng(SEED+777*n).uniform(0, 2*np.pi, size=(M, n))
        res = {}
        for dep in DEPTHS:
            for e in ENTS:
                psi = enc(X, n, dep, e)
                K, _ = kernel_fidelity(psi)
                d = diag_(K); d['mw'] = meyer_wallach(psi, n)
                d['status'], d['a_shot'], d['a_learn'] = layers_(d)
                res[f'{dep}|{e}'] = d
                del psi, K; free_mem()
        json.dump(res, open(p, 'w')); out[n] = res
        r = res[f'{REF["depth"]}|{REF["ent"]}']
        print(f'  fid n={n:2d} M={M} ({time.time()-t0:5.1f}s)  REF: mu={r["offdiag_mean"]:.2e} '
              f'(2^-n={2.0**-n:.2e}) Var={r["offdiag_var"]:.2e} S_req={r["s_req"]:.2e} '
              f'effrank/M={r["effrank_frac"]:.3f} -> {r["status"]}')
    return out

def run_series(H):
    out = {}
    for n in SERIES[H]:
        p = ck(f'H{H}_n{n}')
        if os.path.exists(p):
            out[n] = json.load(open(p)); print(f'  H={H} n={n}: checkpoint'); continue
        t0 = time.time(); M = M_of(n)
        X = np.random.default_rng(SEED+777*n).uniform(0, 2*np.pi, size=(M, n))
        lay = pool_layout(n, H); w = lay[3]; res = {}
        for dep in DEPTHS:
            for e in ENTS:
                psi = enc(X, n, dep, e)
                mw = meyer_wallach(psi, n)
                E = pauli_expect_fast(psi, n, lay)
                del psi; free_mem()
                for pr in PS:
                    K, s2 = kernel_gfqk(E, w, H, pr)
                    d = diag_(K, s2); d['mw'] = mw
                    d['status'], d['a_shot'], d['a_learn'] = layers_(d)
                    res[f'{dep}|{e}|{pr}'] = d
                    del K; free_mem()
                del E; free_mem()
        json.dump(res, open(p, 'w')); out[n] = res
        r = res[f'{REF["depth"]}|{REF["ent"]}|{REF["p"]}']
        print(f'  H={H} n={n:2d} M={M} |havuz|={len(w):6d} ({time.time()-t0:6.1f}s)  REF: '
              f'Var={r["offdiag_var"]:.2e} S_req={r["s_req"]:.2e} '
              f'effrank/M={r["effrank_frac"]:.3f} -> {r["status"]}')
    return out

print('=== A0 FIDELITY REFERANSI ===')
FID = run_fidelity()
SER = {}
for H in sorted(SERIES):
    print(f'\n=== GFQK SERISI  H={H} ===')
    SER[H] = run_series(H)
print('\nKOSU BITTI')

## Bolum 6 - Katman B: egimler ve ustel/polinom testi

In [ ]:
def slopes_for(getter, ns_list, keys):
    out = {}
    for k in keys:
        ns = [n for n in ns_list if k in getter(n)]
        if len(ns) < 3: continue
        out[k] = layer_b(ns, [getter(n)[k]['offdiag_var'] for n in ns])
    return out

fid_keys = [f'{d}|{e}' for d in DEPTHS for e in ENTS]
SL_FID = slopes_for(lambda n: FID[n], N_FID, fid_keys)

print('=== A0 FIDELITY: log Var ~ n egimi  (-ln4 = -1.386 Haar limiti) ===')
print(f'{"":9s}' + ''.join(f'ent={e:<9}' for e in ENTS))
for d in DEPTHS:
    print(f'depth={d:<3} ' + ''.join(f'{SL_FID[f"{d}|{e}"]["slope"]:+9.3f} ' for e in ENTS))
print('\nR2 ustel / polinom (REF kosesi):')
s = SL_FID[f'{REF["depth"]}|{REF["ent"]}']
print(f'  egim={s["slope"]:+.3f} [{s["slope_lo"]:+.3f},{s["slope_hi"]:+.3f}]  '
      f'R2u={s["r2_exp"]:.4f} R2p={s["r2_poly"]:.4f} dAIC={s["d_aic"]:+.1f} USTEL={s["exponential"]}')

print('\n=== GFQK SERILERI: egim H ile nasil degisiyor ===')
SL_SER = {}
for H in sorted(SERIES):
    keys = [f'{d}|{e}|{p}' for d in DEPTHS for e in ENTS for p in PS]
    SL_SER[H] = slopes_for(lambda n, H=H: SER[H][n], SERIES[H], keys)
    k = f'{REF["depth"]}|{REF["ent"]}|{REF["p"]}'
    s = SL_SER[H][k]; nexp = sum(1 for v in SL_SER[H].values() if v and v['exponential'])
    print(f'  H={H} (n<={max(SERIES[H])}): egim={s["slope"]:+.3f} '
          f'[{s["slope_lo"]:+.3f},{s["slope_hi"]:+.3f}] R2u={s["r2_exp"]:.3f} '
          f'R2p={s["r2_poly"]:.3f} USTEL={s["exponential"]}  |  ustel onayli '
          f'{nexp}/{len(SL_SER[H])} konfig')

print('\n=== KATMAN A: hangi n\'de tetikleniyor (REF kosesi) ===')
print(f'{"n":>3} | {"A0 S_req":>10} {"A0 er/M":>8} {"A0 durum":>9} | ' +
      ' | '.join(f'H={H} S_req/er/durum' for H in sorted(SERIES)))
for n in N_FID:
    f_ = FID[n][f'{REF["depth"]}|{REF["ent"]}']
    row = f'{n:>3} | {f_["s_req"]:>10.2e} {f_["effrank_frac"]:>8.3f} {f_["status"]:>9} | '
    for H in sorted(SERIES):
        if n in SER.get(H, {}):
            g = SER[H][n][f'{REF["depth"]}|{REF["ent"]}|{REF["p"]}']
            row += f'{g["s_req"]:.1e} {g["effrank_frac"]:.2f} {g["status"]:<9} | '
        else:
            row += f'{"-":>21} | '
    print(row)

## Bolum 7 - GRAFIKLER

In [ ]:
plt.rcParams.update({'figure.dpi': 120, 'font.size': 9, 'axes.grid': True,
                     'grid.alpha': 0.3, 'figure.autolayout': True})
FIGS = []
def save(fig, name):
    fig.savefig(name, bbox_inches='tight'); FIGS.append(name); print('  ->', name)

rk  = f'{REF["depth"]}|{REF["ent"]}'
rkp = f'{REF["depth"]}|{REF["ent"]}|{REF["p"]}'

# --- F1: log Var vs n -- ANA FIGUR ---
fig, ax = plt.subplots(figsize=(5.2, 3.8))
v = [FID[n][rk]['offdiag_var'] for n in N_FID]
ax.semilogy(N_FID, v, 'o-', c='crimson', lw=2, ms=5, label='A0 fidelity kernel')
nn = np.array(N_FID, float)
ax.semilogy(nn, v[0]*4.0**(-(nn-nn[0])), '--', c='crimson', alpha=.45, lw=1.2,
            label=r'Haar limiti $4^{-n}$')
for H, c in zip(sorted(SERIES), ['#1f77b4', '#2ca02c', '#ff7f0e']):
    ns = SERIES[H]
    ax.semilogy(ns, [SER[H][n][rkp]['offdiag_var'] for n in ns], 's-', c=c, lw=2, ms=4,
                label=f'GFQK $w\\leq{H}$')
ax.set_xlabel('qubit sayisi $n$'); ax.set_ylabel(r'Var$[k(x,x^\prime)]$')
ax.set_title(f'Konsantrasyon: fidelity USTEL, agirlik-kesikli DEGIL\n'
             f'(depth={REF["depth"]}, ent={REF["ent"]}, bw={BW})', fontsize=9)
ax.legend(fontsize=7.5); save(fig, 'F1_variance_vs_n.png')

# --- F2: S_req vs n + butce cizgileri (kesisme) ---
fig, ax = plt.subplots(figsize=(5.2, 3.8))
ax.semilogy(N_FID, [FID[n][rk]['s_req'] for n in N_FID], 'o-', c='crimson', lw=2, ms=5,
            label='A0 fidelity')
ax.semilogy(nn, 2.0**nn, '--', c='crimson', alpha=.45, lw=1.2, label=r'kapali form $2^n$')
for H, c in zip(sorted(SERIES), ['#1f77b4', '#2ca02c', '#ff7f0e']):
    ns = SERIES[H]
    ax.semilogy(ns, [SER[H][n][rkp]['s_req'] for n in ns], 's-', c=c, lw=2, ms=4,
                label=f'GFQK $w\\leq{H}$')
for S, ls in zip(SHOT_BUDGETS, [':', '-', ':']):
    ax.axhline(S, color='k', ls=ls, lw=1.4 if S == LOCK_BUDGET else .8, alpha=.6)
    ax.text(N_FID[0], S*1.4, f'butce {S:.0e}' + (' (KILIT)' if S == LOCK_BUDGET else ''),
            fontsize=6.5, color='k')
ax.set_xlabel('qubit sayisi $n$'); ax.set_ylabel(r'$S_{req}=\sigma^2_{shot}/$Var')
ax.set_title('Olcum maliyeti: kesisme noktasi', fontsize=9)
ax.legend(fontsize=7.5, loc='lower right'); save(fig, 'F2_shots_vs_n.png')

# --- F3: A-learn -- effrank/M vs n ---
fig, ax = plt.subplots(figsize=(5.2, 3.8))
ax.plot(N_FID, [FID[n][rk]['effrank_frac'] for n in N_FID], 'o-', c='crimson', lw=2, ms=5,
        label='A0 fidelity')
for H, c in zip(sorted(SERIES), ['#1f77b4', '#2ca02c', '#ff7f0e']):
    ns = SERIES[H]
    ax.plot(ns, [SER[H][n][rkp]['effrank_frac'] for n in ns], 's-', c=c, lw=2, ms=4,
            label=f'GFQK $w\\leq{H}$')
ax.axhline(EFFRANK_GATE, color='k', ls='--', lw=1.2)
ax.text(N_FID[0], EFFRANK_GATE+.02, f'A-learn esigi {EFFRANK_GATE}', fontsize=7)
ax.set_ylim(0, 1.05); ax.set_xlabel('qubit sayisi $n$')
ax.set_ylabel(r'effrank$(K)/M$')
ax.set_title(r'Ogrenme seviyesinde EC: $K\to I$ (shot\'tan bagimsiz)', fontsize=9)
ax.legend(fontsize=7.5); save(fig, 'F3_effrank_vs_n.png')

# --- F4: eksen atfi -- derinlik x dolaniklik egim isi haritasi ---
Z = np.array([[SL_FID[f'{d}|{e}']['slope'] for e in ENTS] for d in DEPTHS])
fig, ax = plt.subplots(figsize=(4.4, 3.4))
im = ax.imshow(Z, cmap='RdBu_r', vmin=-1.5, vmax=0, aspect='auto')
ax.set_xticks(range(len(ENTS)), [str(e) for e in ENTS])
ax.set_yticks(range(len(DEPTHS)), [str(d) for d in DEPTHS])
ax.set_xlabel('dolaniklik  $ent$'); ax.set_ylabel('derinlik  $depth$')
for i in range(len(DEPTHS)):
    for j in range(len(ENTS)):
        ax.text(j, i, f'{Z[i,j]:.2f}', ha='center', va='center', fontsize=7.5,
                color='w' if Z[i, j] < -0.9 else 'k')
ax.set_title('A0 egimi: derinlik tek basina EC uretmiyor\n(ent=0 sutunu duz)', fontsize=8.5)
ax.grid(False); fig.colorbar(im, ax=ax, label='log Var / n egimi')
save(fig, 'F4_axis_attribution.png')

# --- F5: butceye gore cozulebilir cift orani ---
fig, axs = plt.subplots(1, len(SHOT_BUDGETS), figsize=(9.5, 3.0), sharey=True)
for a, S in zip(axs, SHOT_BUDGETS):
    a.plot(N_FID, [FID[n][rk][f'resolv@{S:.0e}'] for n in N_FID], 'o-', c='crimson',
           lw=1.8, ms=4, label='A0')
    for H, c in zip(sorted(SERIES), ['#1f77b4', '#2ca02c', '#ff7f0e']):
        ns = SERIES[H]
        a.plot(ns, [SER[H][n][rkp][f'resolv@{S:.0e}'] for n in ns], 's-', c=c, lw=1.8,
               ms=3, label=f'$w\\leq{H}$')
    a.set_title(f'butce {S:.0e} shot', fontsize=8.5); a.set_xlabel('$n$'); a.set_ylim(-.03, 1.03)
axs[0].set_ylabel('cozulebilir cift orani'); axs[0].legend(fontsize=7)
save(fig, 'F5_resolvable_fraction.png')
plt.show()

## Bolum 8 - Bolge listesi + indirme

In [ ]:
# --- Katman B kapisi: A-learn EC'ye OZGU DEGIL (carpim durumlarinda da tetikleniyor,
#     n>=14'te 1.0000'da doyuyor). Bolge listesine girmek icin USTEL ONAYI da sart.
def b_ok(depth, ent):
    s_ = SL_FID.get(f'{depth}|{ent}')
    return bool(s_ and s_['exponential'])

regions = []
for n in N_FID:
    for d in DEPTHS:
        for e in ENTS:
            r = FID[n][f'{d}|{e}']
            if r['status'].startswith('EC'):
                regions.append(dict(arm='A0_fidelity', n=n, depth=d, ent=e, p=0.0, H=None,
                                    status=r['status'], s_req=r['s_req'],
                                    effrank_frac=r['effrank_frac'], layer_b=b_ok(d, e)))
for H in sorted(SERIES):
    for n in SERIES[H]:
        for k, r in SER[H][n].items():
            if r['status'].startswith('EC'):
                d, e, p = k.split('|')
                regions.append(dict(arm=f'GFQK_w{H}', n=n, depth=int(d), ent=float(e),
                                    p=float(p), H=H, status=r['status'], s_req=r['s_req'],
                                    effrank_frac=r['effrank_frac'],
                                    layer_b=b_ok(int(d), float(e))))

from collections import Counter
regions_B = [r for r in regions if r['layer_b']]
print(f'=== BOLGE LISTESI: {len(regions)} hucre (A+C) -> {len(regions_B)} (A+B+C, KULLANILACAK) ===')
print(Counter((r['arm'], r['status']) for r in regions_B))
print('B kapisinda elenen:', len(regions)-len(regions_B),
      ' | B gecenlerin ent dagilimi:', dict(sorted(Counter(r['ent'] for r in regions_B).items())))
print('\nEN ONEMLI: A0 EC iken GFQK degil (ayni n, depth, ent) -- ALTIN HUCRELER')
gold = []
for n in N_FID:
    for d in DEPTHS:
        for e in ENTS:
            a0 = FID[n][f'{d}|{e}']
            if not a0['status'].startswith('EC'): continue
            for H in sorted(SERIES):
                if n not in SER.get(H, {}): continue
                g = SER[H][n].get(f'{d}|{e}|0.0')
                if g and g['status'] == 'NO-EC':
                    gold.append(dict(n=n, depth=d, ent=e, H=H, a0_status=a0['status'],
                                     a0_s_req=a0['s_req'], a0_er=a0['effrank_frac'],
                                     g_s_req=g['s_req'], g_er=g['effrank_frac'],
                                     layer_b=b_ok(d, e)))
gold_B = [g for g in gold if g['layer_b']]
print(f'{len(gold)} altin hucre (A+C) -> {len(gold_B)} B-onayli (KULLANILACAK)')
print('  ent=0 (onemsiz EC):', sum(1 for g in gold_B if g['ent'] == 0.0),
      ' | ent>0 (dolaniklikla hizlanan):', sum(1 for g in gold_B if g['ent'] > 0))
for r in sorted(gold_B, key=lambda r: -r['a0_s_req'])[:40]:
    print(f"  n={r['n']:2d} depth={r['depth']} ent={r['ent']:<5} w<={r['H']} | "
          f"A0 {r['a0_status']:8s} S={r['a0_s_req']:.1e} er/M={r['a0_er']:.2f}  ||  "
          f"GFQK S={r['g_s_req']:.1e} er/M={r['g_er']:.2f}")

json.dump(dict(regions=regions, gold=gold, regions_B=regions_B, gold_B=gold_B,
               slopes_fid=SL_FID, slopes_ser={str(k): v for k, v in SL_SER.items()},
               params=dict(SERIES={str(k): v for k, v in SERIES.items()}, N_FID=N_FID,
                           LOCK_BUDGET=LOCK_BUDGET, EFFRANK_GATE=EFFRANK_GATE, BW=BW,
                           MEAN_GATE=MEAN_GATE, DEPTHS=DEPTHS, ENTS=ENTS, PS=PS,
                           REF=REF, SEED=SEED)),
          open('phase_a2_results.json', 'w'), indent=1)
print('\n-> phase_a2_results.json')

import shutil
shutil.make_archive('phase_a2_ckpt', 'zip', CKPT)
try:
    from google.colab import files
    for f in ['phase_a2_ckpt.zip', 'phase_a2_results.json'] + FIGS:
        files.download(f)
except Exception as e:
    print('Colab disi, elle indir:', e)